## Agregación del dataset a nivel de empleado

Pasamos del nivel *evento de ausencia* (805 filas) al nivel *empleado* (136 filas) para evitar la pseudo-replicación y responder "¿qué distingue a un empleado de otro?". Cada variable se agrega según su naturaleza:

**Criterio de agregación por variable:**

| Variable | Agregación | Motivo |
|---|---|---|
| `ID_Absence` → `N_absences` | **count** | Nº de ausencias del empleado (es un identificador, no se suma). |
| `Reason_Absence`, `Month`, `Day_Week`, `Seasons` | **moda** | Categóricas/temporales: el valor más frecuente representa el patrón habitual. |
| `Absenteeism_Hours` → `Absent_median` | **mediana** | Duración típica de una ausencia; robusta al fuerte sesgo a la derecha (la media se distorsiona por outliers). Se guarda también la suma total. |
| `Work_load_Average_Day`, `Hit_Target` | **media** | Nivel medio de carga y de desempeño del empleado. |
| `Disciplinary_Failure` → `N_disciplinary` | **suma** | La falta va ligada a la *solicitud de ausencia* (varía entre filas del mismo empleado), no a la persona → se cuentan cuántas tuvo. |
| Edad, antigüedad, distancia, transporte, educación, hijos, hábitos, mascota | **max** (≡ valor único) | Son **constantes por empleado** (verificado): max, mediana o first dan el mismo resultado. |

El resultado es `emp`: un registro por empleado, base para el perfilado top vs bottom y las regresiones exploratorias.

In [5]:
import pandas as pd, numpy as np

df = pd.read_csv(r"C:\Users\User\Desktop\SIMULADOR\Sprint2\RRHH_CLEAN_07072026.csv")

def moda(s):
    """Valor más frecuente; robusto a NaN y a empates (devuelve el primero)."""
    s = s.dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan

emp = df.groupby('ID_Employee').agg(
    # --- Absentismo y temporalidad (nivel evento) ---
    N_absences        = ('ID_Absence', 'count'),          # nº de ausencias (count, es un ID)
    Reason_moda       = ('Reason_Absence', moda),         # motivo más frecuente
    Month_moda        = ('Month_Absence', moda),
    DayWeek_moda      = ('Day_Week', moda),
    Season_moda       = ('Seasons', moda),
    Absent_median     = ('Absenteeism_Hours', 'median'),  # duración típica (robusta al sesgo)
    Absent_total      = ('Absenteeism_Hours', 'sum'),     # carga total (por si se necesita)

    # --- Variables que varían por empleado ---
    Work_load_mean    = ('Work_load_Average_Day', 'mean'),
    Hit_Target_mean   = ('Hit_Target', 'mean'),
    N_disciplinary    = ('Disciplinary_Failure', 'sum'),  # nº de faltas (ligadas a la solicitud)

    # --- Constantes por empleado (max = first = mismo valor único) ---
    Transport_med     = ('Transportation_Expense', 'median'),
    Distance_med      = ('Distance_Residence_Work', 'median'),
    Service_Time      = ('Service_Time', 'max'),
    Age               = ('Age', 'max'),
    Education          = ('Education', 'max'),
    Son               = ('Son', 'max'),
    Social_Drinker    = ('Social_Drinker', 'max'),
    Social_Smoker     = ('Social_Smoker', 'max'),
    Pet               = ('Pet', 'max'),
).reset_index()

# --- Comprobaciones de robustez ---
assert emp['ID_Employee'].is_unique, "Hay IDs de empleado duplicados"
print("Empleados:", emp.shape[0], "| variables:", emp.shape[1])
print("Nulos por columna:\n", emp.isna().sum()[emp.isna().sum() > 0] if emp.isna().any().any() else "  sin nulos")
emp.head()

# Eliminar columnas relativas al IMC tras la agregación
emp = emp.drop(columns=['Body_Mass_Index', 'Weight', 'Height'], errors='ignore')

print("Variables tras el drop:", emp.shape[1])
print(emp.columns.tolist())

emp


Empleados: 136 | variables: 20
Nulos por columna:
   sin nulos
Variables tras el drop: 20
['ID_Employee', 'N_absences', 'Reason_moda', 'Month_moda', 'DayWeek_moda', 'Season_moda', 'Absent_median', 'Absent_total', 'Work_load_mean', 'Hit_Target_mean', 'N_disciplinary', 'Transport_med', 'Distance_med', 'Service_Time', 'Age', 'Education', 'Son', 'Social_Drinker', 'Social_Smoker', 'Pet']


,ID_Employee,N_absences,Reason_moda,Month_moda,DayWeek_moda,Season_moda,Absent_median,Absent_total,Work_load_mean,Hit_Target_mean,N_disciplinary,Transport_med,Distance_med,Service_Time,Age,Education,Son,Social_Drinker,Social_Smoker,Pet
0,1,23,22,8,2,3,4.0,121,262.894478,95.173913,1,235.0,11.0,14,37,3,1,0,0,1
1,2,6,0,8,2,3,4.5,25,241.597000,93.333333,2,235.0,29.0,12,48,1,1,0,1,5
2,3,97,28,2,2,1,3.0,444,264.747216,94.762887,1,179.0,51.0,18,38,1,0,1,0,0
3,4,1,0,0,3,0,0.0,0,271.219000,95.000000,0,118.0,14.0,13,40,1,1,1,0,8
4,5,18,26,9,2,4,8.0,102,266.741389,91.722222,5,235.0,20.0,13,43,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,132,1,0,5,4,2,0.0,0,237.656000,99.000000,1,378.0,49.0,11,36,1,2,0,1,4
132,133,1,25,3,2,2,3.0,3,222.196000,99.000000,0,235.0,16.0,8,32,3,0,0,0,0
133,134,1,23,12,4,1,2.0,2,261.306000,97.000000,0,118.0,13.0,18,50,1,1,1,0,0
134,135,1,21,11,4,4,8.0,8,306.345000,93.000000,0,179.0,22.0,17,40,2,2,0,1,0


In [6]:
import statsmodels.api as sm
from statsmodels.othermod.betareg import BetaModel

# Target -> proporción (0,1)
n = len(emp)
y = emp['Hit_Target_mean'] / 100.0
emp['perf'] = (y*(n-1) + 0.5) / n

# Estandarizar continuas (sin IMC)
cont = ['Work_load_mean','Absent_median','N_disciplinary','N_absences',
        'Age','Service_Time','Distance_med','Transport_med']
Z = emp.copy()
for c in cont:
    Z[c] = (emp[c] - emp[c].mean()) / emp[c].std()

# Regresión Beta (sin Body_Mass_Index)
formula = ('perf ~ Work_load_mean + Absent_median + N_disciplinary + N_absences + Age '
           '+ Service_Time + Distance_med + Transport_med + Son + Pet + Social_Drinker '
           '+ Social_Smoker + C(Education)')
res = BetaModel.from_formula(formula, Z, link_precision=sm.families.links.Log()).fit(maxiter=500, disp=0)
print('pseudoR2:', round(res.prsquared, 4), '| empleados:', n)
print(res.summary())

# Ranking de atributos
r = [(nm, res.params[nm], np.exp(res.params[nm]), res.tvalues[nm], res.pvalues[nm])
     for nm in res.params.index if nm != 'Intercept' and not nm.startswith('precision')]
rk = pd.DataFrame(r, columns=['var','coef','OR','z','p'])
rk['|z|'] = rk['z'].abs()
rk['sig'] = np.where(rk['p']<0.05, '**', np.where(rk['p']<0.10, '*', ''))
rk.sort_values('|z|', ascending=False).reset_index(drop=True).round(4)

pseudoR2: 0.1854 | empleados: 136
                              BetaModel Results                               
Dep. Variable:                   perf   Log-Likelihood:                 312.65
Model:                      BetaModel   AIC:                            -591.3
Method:            Maximum Likelihood   BIC:                            -541.8
Date:                Wed, 08 Jul 2026                                         
Time:                        13:58:19                                         
No. Observations:                 136                                         
Df Residuals:                     119                                         
Df Model:                          15                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             3.0045      0.109     27.622      0.000       2.791       3.2

,var,coef,OR,z,p,|z|,sig
0,Work_load_mean,-0.1278,0.8801,-3.0397,0.0024,3.0397,**
1,N_disciplinary,-0.0835,0.9199,-1.8532,0.0639,1.8532,*
2,Social_Drinker,-0.1917,0.8255,-1.6583,0.0973,1.6583,*
3,C(Education)[T.4],0.6583,1.9315,1.2938,0.1957,1.2938,
4,Pet,0.0483,1.0495,1.2864,0.1983,1.2864,
5,Age,0.0730,1.0758,1.0841,0.2783,1.0841,
6,C(Education)[T.3],-0.1658,0.8473,-0.9804,0.3269,0.9804,
7,Social_Smoker,-0.1386,0.8706,-0.9596,0.3372,0.9596,
8,Transport_med,0.0505,1.0518,0.9090,0.3634,0.9090,
9,Service_Time,0.0651,1.0673,0.8587,0.3905,0.8587,


## Conclusiones — Regresión Beta a nivel empleado (sin IMC)

**Dataset:** 136 empleados · **Target:** `Hit_Target_mean` (desempeño medio) · **Modelo:** Beta, enlace logit.

**Hallazgo robusto:**
- **Carga de trabajo (`Work_load_mean`)** — único predictor significativo (OR 0.88, p=0.0024), negativo. Aparece en TODOS los cortes del análisis (nivel evento y nivel empleado) → es la conclusión más sólida y fiable del proyecto: a mayor carga media, menor desempeño.

**Señales débiles (marginales, tratar con cautela):**
- `N_disciplinary` (OR 0.92, p=0.064): apunta negativo; recupera algo de señal con la agregación por suma, pero no es concluyente.
- `Social_Drinker` (OR 0.83, p=0.097): negativo y frágil.

**Sin relación con el desempeño:**
- **Absentismo** (`Absent_median` p=0.96, `N_absences` p=0.67): cero asociación, reconfirmado.
- Edad, antigüedad, distancia, transporte, hijos, mascota, educación: no significativos.
- `Social_Smoker` perdió la significancia al eliminar el IMC → su efecto previo era un artefacto compartido con el IMC (ruido).

**Lectura de fondo:** más allá de la carga de trabajo, no existe un perfil personal robusto que prediga el desempeño. Las señales secundarias aparecen y desaparecen según la especificación del modelo, lo que indica que son inestables. La única palanca fiable y accionable para negocio es el **equilibrio de la carga de trabajo**. Todo ello es asociación, no causalidad.

## Regresión Beta sobre el Índice de Eficiencia Relativa (IER)

**Cambio de variable dependiente:** en lugar de `Hit_Target` usamos el KPI **IER = Σ(Hit_Target) / Σ(Work_load_Average_Day)**, que a nivel empleado equivale a `Hit_Target_mean / Work_load_mean`. Mide el cumplimiento de objetivos **por unidad de carga soportada**.

**Por qué es válida la Beta:** el IER de nuestros empleados está en el rango 0.24–0.45, es decir, dentro del intervalo (0,1) que exige la distribución Beta. (Si el índice pudiera superar 1, habría que usar regresión Gamma.)

**Cautela clave — evitar circularidad:** tanto `Hit_Target` (numerador) como `Work_load` (denominador) forman parte del IER. Incluir cualquiera de los dos como predictor daría una correlación tautológica. **Ambos se excluyen** del modelo. Analizamos solo atributos de la persona.

**Pregunta:** ¿qué características personales distinguen a los empleados más eficientes (más cumplimiento por unidad de carga)?

In [7]:
import numpy as np, pandas as pd
import statsmodels.api as sm
from statsmodels.othermod.betareg import BetaModel

# 1. Construir el IER (= media_hit / media_carga, equivalente a Σhit/Σcarga)
emp['IER'] = emp['Hit_Target_mean'] / emp['Work_load_mean']

# 2. Comprobar que el IER cae en (0,1) -> requisito de la Beta
print("IER rango:", round(emp['IER'].min(), 3), "-", round(emp['IER'].max(), 3))
assert emp['IER'].between(0, 1, inclusive='neither').all(), "Hay IER fuera de (0,1): usar Gamma"

# 3. Estandarizar continuas (SIN Work_load ni Hit_Target: están en la fórmula del IER)
cont = ['Absent_median','N_disciplinary','N_absences','Age','Service_Time',
        'Distance_med','Transport_med']
Z = emp.copy()
for c in cont:
    Z[c] = (emp[c] - emp[c].mean()) / emp[c].std()

# 4. Regresión Beta sobre IER (excluye Work_load y Hit_Target)
formula = ('IER ~ Absent_median + N_disciplinary + N_absences + Age + Service_Time '
           '+ Distance_med + Transport_med + Son + Pet + Social_Drinker '
           '+ Social_Smoker + C(Education)')
res = BetaModel.from_formula(formula, Z, link_precision=sm.families.links.Log()).fit(maxiter=500, disp=0)
print('pseudoR2:', round(res.prsquared, 4), '| empleados:', len(emp))
print(res.summary())

# 5. Ranking de atributos
r = [(nm, res.params[nm], np.exp(res.params[nm]), res.tvalues[nm], res.pvalues[nm])
     for nm in res.params.index if nm != 'Intercept' and not nm.startswith('precision')]
rk = pd.DataFrame(r, columns=['var','coef','OR','z','p'])
rk['|z|'] = rk['z'].abs()
rk['sig'] = np.where(rk['p']<0.05, '**', np.where(rk['p']<0.10, '*', ''))
rk.sort_values('|z|', ascending=False).reset_index(drop=True).round(4)

IER rango: 0.243 - 0.447
pseudoR2: 0.0707 | empleados: 136
                              BetaModel Results                               
Dep. Variable:                    IER   Log-Likelihood:                 240.63
Model:                      BetaModel   AIC:                            -449.3
Method:            Maximum Likelihood   BIC:                            -402.7
Date:                Thu, 09 Jul 2026                                         
Time:                        11:12:13                                         
No. Observations:                 136                                         
Df Residuals:                     120                                         
Df Model:                          14                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.6238      0.036    -17.172      0.

,var,coef,OR,z,p,|z|,sig
0,C(Education)[T.4],0.2453,1.2781,1.8719,0.0612,1.8719,*
1,C(Education)[T.3],0.1017,1.1070,1.7328,0.0831,1.7328,*
2,Transport_med,-0.0234,0.9769,-1.2104,0.2261,1.2104,
3,Age,0.0262,1.0265,1.1270,0.2597,1.1270,
4,Distance_med,0.0222,1.0225,1.1029,0.2701,1.1029,
5,Service_Time,-0.0221,0.9782,-0.8928,0.3720,0.8928,
6,Absent_median,0.0148,1.0149,0.8868,0.3752,0.8868,
7,Social_Smoker,0.0420,1.0429,0.8031,0.4219,0.8031,
8,Social_Drinker,0.0261,1.0265,0.6574,0.5109,0.6574,
9,C(Education)[T.2],0.0392,1.0400,0.5840,0.5592,0.5840,


### Interpretación — Resultados de la Beta sobre el IER

**Poder explicativo bajo:** pseudo-R² = 0.07. Con la carga de trabajo fuera del modelo (por estar en la fórmula del IER), los atributos personales apenas explican un 7% de la eficiencia.

**Ninguna variable es concluyente.** Ningún atributo alcanza significancia al 5%. Lo único que asoma, de forma **marginal**, es el nivel educativo:

| Variable | OR | p | |
|---|---:|---:|:--:|
| Education [nivel 4] | 1.278 | 0.061 | `*` |
| Education [nivel 3] | 1.107 | 0.083 | `*` |
| resto de variables | — | > 0.22 | |

- **Educación (niveles 3 y 4):** apunta a que un mayor nivel educativo se asocia con algo más de eficiencia (OR > 1). Es la única señal, y débil (marginal). Coincide con lo observado en la versión Gamma, lo que le da cierta credibilidad como pista, no como prueba.
- **Todo lo demás es ruido:** absentismo (`Absent_median`, `N_absences`), disciplina (`N_disciplinary`), edad, antigüedad, distancia, transporte, hábitos, hijos y mascota, todos con p muy altas (> 0.22).

**Lectura de fondo:** cuando el desempeño se mide **ajustado por la carga de trabajo** (que es lo que hace el IER), el **perfil personal del empleado prácticamente no lo explica**. Los empleados eficientes e ineficientes apenas se distinguen por sus características; a lo sumo, la formación da una ligera ventaja. Refuerza la conclusión transversal del estudio: el rendimiento no está escrito en el perfil demográfico del empleado, sino que depende principalmente de la carga (factor de gestión) y, con toda cautela, de la formación. Asociación, no causalidad.

## Regresión Gamma sobre el Índice de Eficiencia Relativa (IER)

**Por qué también Gamma:** ya modelamos el IER con Beta (válida porque cae en (0,1)). Corremos además una **regresión Gamma con enlace log** como comprobación de robustez: el IER es un *ratio positivo*, y la Gamma es la distribución natural para ese tipo de variable. Si ambos modelos (Beta y Gamma) coinciden, la conclusión es más sólida.

**Mismo criterio anti-circularidad:** se excluyen `Work_load` y `Hit_Target`, que forman parte de la fórmula del IER. Analizamos solo atributos de la persona.

**Pregunta:** ¿qué características personales distinguen a los empleados más eficientes?

In [8]:
import numpy as np, pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# IER (= media_hit / media_carga). Si ya existe, esta línea es redundante.
emp['IER'] = emp['Hit_Target_mean'] / emp['Work_load_mean']

# Estandarizar continuas (SIN Work_load ni Hit_Target: están en la fórmula del IER)
cont = ['Absent_median','N_disciplinary','N_absences','Age','Service_Time',
        'Distance_med','Transport_med']
Z = emp.copy()
for c in cont:
    Z[c] = (emp[c] - emp[c].mean()) / emp[c].std()

# Regresión Gamma con enlace log
formula = ('IER ~ Absent_median + N_disciplinary + N_absences + Age + Service_Time '
           '+ Distance_med + Transport_med + Son + Pet + Social_Drinker '
           '+ Social_Smoker + C(Education)')
m = smf.glm(formula, Z, family=sm.families.Gamma(link=sm.families.links.Log())).fit()
print('Pseudo-R2:', round(1 - m.deviance/m.null_deviance, 4), '| empleados:', len(emp))
print(m.summary())

# Ranking de atributos
r = [(nm, m.params[nm], np.exp(m.params[nm]), m.tvalues[nm], m.pvalues[nm])
     for nm in m.params.index if nm != 'Intercept']
rk = pd.DataFrame(r, columns=['var','coef','exp_coef','z','p'])
rk['|z|'] = rk['z'].abs()
rk['sig'] = np.where(rk['p']<0.05, '**', np.where(rk['p']<0.10, '*', ''))
rk.sort_values('|z|', ascending=False).reset_index(drop=True).round(4)

Pseudo-R2: 0.0679 | empleados: 136
                 Generalized Linear Model Regression Results                  
Dep. Variable:                    IER   No. Observations:                  136
Model:                            GLM   Df Residuals:                      121
Model Family:                   Gamma   Df Model:                           14
Link Function:                    Log   Scale:                        0.014705
Method:                          IRLS   Log-Likelihood:                 238.99
Date:                Thu, 09 Jul 2026   Deviance:                       1.8630
Time:                        12:48:44   Pearson chi2:                     1.78
No. Iterations:                     9   Pseudo R-squ. (CS):            0.06560
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Int

,var,coef,exp_coef,z,p,|z|,sig
0,C(Education)[T.4],0.1481,1.1596,1.6347,0.1021,1.6347,
1,C(Education)[T.3],0.0642,1.0663,1.6136,0.1066,1.6136,
2,Transport_med,-0.0152,0.9849,-1.1560,0.2477,1.1560,
3,Age,0.0168,1.0170,1.0750,0.2824,1.0750,
4,Distance_med,0.0137,1.0138,0.9997,0.3175,0.9997,
5,Service_Time,-0.0141,0.9860,-0.8404,0.4007,0.8404,
6,Absent_median,0.0089,1.0090,0.7949,0.4267,0.7949,
7,Social_Smoker,0.0263,1.0266,0.7435,0.4572,0.7435,
8,Social_Drinker,0.0184,1.0185,0.6832,0.4945,0.6832,
9,C(Education)[T.2],0.0272,1.0276,0.5998,0.5486,0.5998,


### Interpretación — Resultados de la Gamma sobre el IER

**Confirma la Beta: ninguna variable es concluyente.** El modelo Gamma (comprobación de robustez del IER) da el mismo mensaje que la Beta. Aquí **ninguna variable alcanza siquiera la significancia marginal** (todas con p > 0.10):

| Variable | exp(coef) | p | |
|---|---:|---:|:--:|
| Education [nivel 4] | 1.160 | 0.102 | (casi) |
| Education [nivel 3] | 1.066 | 0.107 | (casi) |
| resto de variables | — | > 0.24 | |

- **Educación (niveles 3 y 4):** vuelve a ser lo único que se acerca, con el mismo signo positivo que en la Beta (OR > 1), pero esta vez **se queda justo por debajo del umbral** (p ≈ 0.10–0.11). Es decir, apenas una pista, ni siquiera marginal.
- **Todo lo demás, ruido:** absentismo, disciplina, edad, antigüedad, distancia, transporte, hábitos, hijos y mascota, todos claramente no significativos (p > 0.24).

**Coincidencia Beta ↔ Gamma:** los dos modelos, con distribuciones distintas (Beta para proporciones acotadas, Gamma para ratios positivos), **coinciden en el resultado**: la educación es la única señal que asoma, siempre débil y en el límite, y ninguna otra característica personal se relaciona con la eficiencia. Que dos métodos independientes lleguen a lo mismo blinda la conclusión.

**Lectura de fondo:** ajustando el desempeño por la carga de trabajo (IER), el **perfil personal del empleado no explica la eficiencia**. No hay un "perfil de empleado eficiente" identificable a partir de estos datos. Consistente con toda la batería de análisis del estudio: el rendimiento depende de la carga (gestión) y, con máxima cautela, de la formación. Asociación, no causalidad.

# Conclusiones generales — Análisis de Perfil y Desempeño

## Qué hemos hecho

Para responder qué características del empleado se asocian a un mejor desempeño, seguimos una estrategia progresiva y cada vez más rigurosa:

1. **Agregación a nivel de empleado.** Pasamos de 805 eventos de ausencia a 136 empleados, agregando cada variable según su naturaleza (media, suma, moda, count), para evitar la pseudo-replicación y responder la pregunta correcta: *"¿qué distingue a un empleado de otro?"*.

2. **Regresión Beta — Empleado vs. Hit_Target.** Modelamos el desempeño medio (`Hit_Target_mean`) como proporción acotada. El único factor con algo de solidez fue la **carga de trabajo** (asociación negativa). El resto de atributos personales, sin señal fiable, y las pocas marginales (disciplina, bebedor) inestables según la especificación del modelo.

3. **Regresión Beta y Gamma — Empleado vs. Índice de Eficiencia Relativa (IER).** Cambiamos la variable dependiente por un KPI que corrige el desempeño por la carga soportada (IER = Σhit / Σcarga), excluyendo del modelo las variables que lo componen para evitar circularidad. **Ambos modelos coinciden:** ninguna característica personal es concluyente; a lo sumo, el nivel educativo asoma de forma marginal (y en la Gamma, ni eso).

## Resultado global

El estudio **no arroja resultados claros ni concluyentes** sobre qué perfil de empleado rinde mejor. Más allá de la carga de trabajo —un factor de gestión, no de perfil— las características personales del empleado **no predicen el desempeño de forma fiable**. Es un resultado consistente en todos los modelos (Beta sobre Hit_Target, Beta y Gamma sobre IER) y coherente con el de los equipos anteriores.

## Nota metodológica: por qué el "nada" es un resultado sólido

Desde las primeras pruebas ya se intuía que **no saldrían resultados concluyentes**, por dos motivos de fondo: la **baja variabilidad del Hit_Target** (casi todos los empleados entre 81 y 100, concentrados en torno a 95) y el **reducido tamaño de muestra** (136 empleados). Ante ese escenario, en lugar de forzar conclusiones, optamos por **blindarnos con buenas prácticas**:

- **Agregar a nivel de empleado** en lugar de analizar eventos de ausencia (evitando la pseudo-replicación que inflaba artificialmente la muestra).
- **Trabajar con el IER** en lugar del Hit_Target crudo, para medir el desempeño ajustado por la carga y no confundir "rendir bien" con "tener poca carga".

Así garantizamos que la **ausencia de hallazgos no se debe a un análisis mal planteado**, sino que refleja la realidad de los datos: el rendimiento depende principalmente de la carga de trabajo (gestión) y, con máxima cautela, de la formación. **Todo ello es asociación, no causalidad.**